# Image Segmentation: Split and Merge

This notebook documents a top-down approach to digital image segmentation. Unlike region-growing algorithms (which start from a local seed), the Split and Merge method begins with the full image, recursively decomposes it, and then merges the extracted regions.

The project covers:
- **Quad-tree decomposition:** Recursion splits regions into four equal quadrants when they fail a homogeneity predicate (expressed as color variance standard deviation).
- **HSV color space analysis:** Mapping the image to a cylindrical color space and isolating the Hue channel to decouple segmentation from uneven illumination.
- **Neighborhood detection via mathematical morphology:** Dilation to derive region boundaries and identify adjacent regions.
- **Object-oriented encapsulation (OOP):** Elimination of global state in favor of managing the segmentation pipeline inside an optimized class.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import glob

# Load and prepare input data
if not os.path.exists("umbrella.png") :
    !wget -q https://raw.githubusercontent.com/vision-agh/poc_sw/master/12_Segmentation/umbrella.png --no-check-certificate

# Color space conversion: BGR to standardized RGB and HSV
img_rgb = cv2.cvtColor(cv2.imread('umbrella.png'), cv2.COLOR_BGR2RGB)
img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

# Extract normalized H (Hue) channel
H_channel = img_hsv[:,:,0].astype(float) / 255.0

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_rgb)
plt.title("Original RGB Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(H_channel, cmap='gray')
plt.title("Normalized Hue Channel")
plt.axis('off')
plt.show()

## Segmentation Class Architecture (Split & Merge)
Encapsulating logic inside a class prevents memory leaks and index collisions in deep recursive calls (removing anti-patterns that rely on the `global` keyword).

In [ ]:
class SplitAndMergeSegmenter:
    def __init__(self, image_channel, std_threshold=0.05, min_block_size=8):
        self.img = image_channel
        self.h, self.w = image_channel.shape
        self.std_th = std_threshold
        self.min_size = min_block_size
        
        # Initialize output matrices
        self.seg_res = np.zeros((self.h, self.w), dtype=np.int16)
        self.m_res = np.zeros((self.h, self.w), dtype=float)
        self.current_idx = 1
        
    def split_phase(self, y1, y2, x1, x2):
        """Recursive split phase (quad-tree decomposition)."""
        fragment = self.img[y1:y2, x1:x2]
        m, s = np.mean(fragment), np.std(fragment)
        
        # Leaf condition: variance homogeneity OR minimum region size reached
        if s < self.std_th or (y2 - y1) <= self.min_size or (x2 - x1) <= self.min_size:
            self.seg_res[y1:y2, x1:x2] = self.current_idx
            self.m_res[y1:y2, x1:x2] = m
            self.current_idx += 1
        else:
            # Split region into 4 quadrants and recurse
            hy, hx = (y1 + y2) // 2, (x1 + x2) // 2
            self.split_phase(y1, hy, x1, hx) # Top-left
            self.split_phase(y1, hy, hx, x2) # Top-right
            self.split_phase(hy, y2, x1, hx) # Bottom-left
            self.split_phase(hy, y2, hx, x2) # Bottom-right

    def merge_phase(self, merge_threshold=5.0/255.0):
        """Merge phase via boundary detection through morphological dilation."""
        i = 1
        kernel = np.ones((3, 3), np.uint8)
        
        while i <= self.current_idx:
            ib = (self.seg_res == i).astype(np.uint8)
            # Skip regions merged in previous iterations
            if not np.any(ib):
                i += 1
                continue
                
            y, x = np.nonzero(ib)
            fy, fx = y[0], x[0]
            
            # Extract boundary outline and find neighboring region indices
            dilated = cv2.dilate(ib, kernel)
            boundaries = (dilated - ib) * self.seg_res
            neighbors = np.unique(boundaries[boundaries > 0])
            
            merged = False
            for neighbor_idx in neighbors:
                ibs = (self.seg_res == neighbor_idx).astype(np.uint8)
                sy, sx = np.nonzero(ibs)
                if len(sy) == 0: 
                    continue
                
                # Merge predicate: convergence of mean Hue in adjacent regions
                if abs(self.m_res[fy, fx] - self.m_res[sy[0], sx[0]]) < merge_threshold:
                    self.seg_res[ibs == 1] = i
                    merged = True
                    
            # Increment main index only when the region has no further merge candidates
            if not merged:
                i += 1
                
    def filter_and_reindex(self, min_area=100):
        """Remove regions below significance threshold and reindex labels continuously."""
        for unique_id in np.unique(self.seg_res):
            mask = (self.seg_res == unique_id)
            if np.sum(mask) < min_area:
                self.seg_res[mask] = 0
                
        for new_id, unique_id in enumerate(np.unique(self.seg_res)):
            self.seg_res[self.seg_res == unique_id] = new_id

## Segmentation Pipeline Execution

In [ ]:
# 1. Initialize segmenter with H channel
segmenter = SplitAndMergeSegmenter(H_channel, std_threshold=0.05, min_block_size=8)

# 2. Split phase (quad-tree decomposition)
segmenter.split_phase(0, segmenter.h, 0, segmenter.w)

plt.figure(figsize=(15, 6))
plt.subplot(1, 3, 1)
plt.imshow(segmenter.seg_res, cmap='nipy_spectral')
plt.title("Stage 1: After Split Decomposition")
plt.axis('off')

# 3. Merge phase
segmenter.merge_phase(merge_threshold=5.0/255.0)

plt.subplot(1, 3, 2)
plt.imshow(segmenter.seg_res, cmap='nipy_spectral')
plt.title("Stage 2: After Merge Phase")
plt.axis('off')

# 4. Filter phase
segmenter.filter_and_reindex(min_area=100)

plt.subplot(1, 3, 3)
plt.imshow(segmenter.seg_res, cmap='nipy_spectral')
plt.title("Stage 3: Final Segmentation (Noise Removed)")
plt.axis('off')
plt.tight_layout()
plt.show()

### Workspace Cleanup

In [ ]:
trash_files = glob.glob('umbrella.png')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

print("Temporary test images removed from the filesystem.")